# Scenario 3: Customer Segmentation Query
Scenario: The product team needs to segment customers based on their purchasing behavior for a new feature rollout.

Database Schema:

user_activity
 user_id
last_login_date
feature_usage_count
account_type
transactions
 transaction_id
user_id
transaction_date
amount
platform
user_preferences
 user_id
communication_preference
interface_theme
notification_settings
Task: Create a SQL query to identify:

Active users (logged in last 30 days)
Filter by high-value customers (top 20% by spending)
User preference trends for the identified customers
Discussion Points:

How were percentile calculations handled?
What approaches to date filtering were used?
How was the query optimized, CTE, subquery etc…?

## AI Prompt

Act as a senior SQL developer and data analyst.

I am working on customer segmentation for a product rollout. I need an optimized SQL query using best practices and clear documentation.

Database schema:

Table: user_activity
- user_id
- last_login_date
- feature_usage_count
- account_type

Table: transactions
- transaction_id
- user_id
- transaction_date
- amount
- platform

Table: user_preferences
- user_id
- communication_preference
- interface_theme
- notification_settings

Task:
Create a SQL query that:

1. Identifies active users who logged in within the last 30 days
2. Identifies high-value customers based on the top 20% of total spending
3. Analyzes user preference trends for those customers
4. Uses optimized SQL practices such as CTEs and window functions
5. Includes comments explaining the logic
6. Handles NULL values and duplicate records where necessary

Please optimize for readability and performance.

## Generated SQL Query

In [ ]:
#---------------------------------------
# Customer Segmentation Analysis
#---------------------------------------

WITH active_users AS (

    # Users active in the last 30 days
    SELECT
        DISTINCT user_id,
        last_login_date,
        feature_usage_count,
        account_type
    FROM user_activity
    WHERE last_login_date >= CURRENT_DATE - INTERVAL '30 days'

),

customer_spending AS (

    # Calculate total spending per user
    SELECT
        user_id,
        SUM(amount) AS total_spending
    FROM transactions
    GROUP BY user_id

),

ranked_customers AS (

    # Rank customers by spending
    SELECT
        user_id,
        total_spending,
        NTILE(5) OVER (
            ORDER BY total_spending DESC
        ) AS spending_percentile
    FROM customer_spending

),

high_value_customers AS (

    # Top 20% customers
    SELECT
        user_id,
        total_spending
    FROM ranked_customers
    WHERE spending_percentile = 1

)

SELECT
    au.user_id,
    au.account_type,
    au.feature_usage_count,
    hvc.total_spending,
    up.communication_preference,
    up.interface_theme,
    up.notification_settings

FROM active_users au

INNER JOIN high_value_customers hvc
    ON au.user_id = hvc.user_id

LEFT JOIN user_preferences up
    ON au.user_id = up.user_id

ORDER BY hvc.total_spending DESC;

## Follow-up Prompts

1. Optimize the query for large datasets with millions of records.
2. Replace NTILE with PERCENT_RANK for more accurate percentile calculations.
3. Add platform-level spending segmentation.
4. Include monthly transaction trends.
5. Explain the performance benefits of using CTEs versus nested subqueries.
6. Add handling for NULL transaction amounts and duplicate transactions.

## Final Solution (Refined Version)

In [ ]:
# ==================================================
# Customer Segmentation for Feature Rollout
# ==================================================

WITH cleaned_transactions AS (

    # Remove invalid transactions
    SELECT DISTINCT
        transaction_id,
        user_id,
        transaction_date,
        COALESCE(amount, 0) AS amount,
        platform
    FROM transactions
    WHERE amount IS NOT NULL
),

active_users AS (

    # Users active within the last 30 days
    SELECT DISTINCT
        user_id,
        last_login_date,
        feature_usage_count,
        account_type
    FROM user_activity
    WHERE last_login_date >= CURRENT_DATE - INTERVAL '30 days'
),

customer_spending AS (

    # Total spending per customer
    SELECT
        user_id,
        SUM(amount) AS total_spending
    FROM cleaned_transactions
    GROUP BY user_id
),

ranked_customers AS (

    # Percentile ranking by spending
    SELECT
        user_id,
        total_spending,

        PERCENT_RANK() OVER (
            ORDER BY total_spending
        ) AS spending_rank

    FROM customer_spending
),

high_value_customers AS (

    # Top 20% spenders
    SELECT
        user_id,
        total_spending
    FROM ranked_customers
    WHERE spending_rank >= 0.80
),

preference_analysis AS (

    # User preference insights
    SELECT
        up.user_id,
        up.communication_preference,
        up.interface_theme,
        up.notification_settings
    FROM user_preferences up
)

SELECT
    au.user_id,
    au.account_type,
    au.feature_usage_count,
    hvc.total_spending,
    pa.communication_preference,
    pa.interface_theme,
    pa.notification_settings

FROM active_users au

INNER JOIN high_value_customers hvc
    ON au.user_id = hvc.user_id

LEFT JOIN preference_analysis pa
    ON au.user_id = pa.user_id

ORDER BY hvc.total_spending DESC;

## Discussion Questions

1. How were percentile calculations handled?

Different prompts handled percentile calculations using different SQL window functions. Some prompts used NTILE(5) to divide customers into five equal groups, where the top group represented the top 20% of spenders. More advanced prompts used PERCENT_RANK() for more precise percentile calculations because it provides a continuous ranking scale between 0 and 1.

For example:

PERCENT_RANK() OVER (ORDER BY total_spending)

This approach improved segmentation accuracy for identifying high-value customers.

2. What approaches to date filtering were used?

Most prompts used relative date filtering with:

CURRENT_DATE - INTERVAL '30 days'

This dynamically identifies users who logged in within the last 30 days. Some prompts also suggested using indexed date columns and partitioning large tables by date to improve query performance in production environments.

3. How was the query optimized (CTE, subquery, etc.)?

Different optimization strategies included:

Using Common Table Expressions (CTEs) to separate logic into readable steps
Reducing duplicate records with DISTINCT
Handling NULL values using COALESCE
Using window functions instead of multiple nested subqueries
Aggregating transaction data before joins to reduce processing load

CTEs improved readability and debugging, while window functions improved performance and scalability for customer segmentation tasks.